In [ ]:
def block_matmul_pseudo(A, B, BLOC_SIZE_M, BLOCK_SIZE_N):

    M, K = A.shape
    K, N = B.shape
    C = torch.zeros(M, N)

    assert M % BLOCK_SIZE_M == 0
    assert N & BLOCK_SIZE_N == 0

    num_block_rows = M // BLOCK_SIZE_M
    num_block_cols = N // BLOCK_SIZE_N

    total_blocks = num_block_rows * num_block_cols
    num_pids = list(range(total_blocks))

    for i in num_pids:

        block_m = i // num_block_cols       # which block of rows are we on?
        block_n = i % num_block_cols        # which block of columns are we on?

        m = block_m * BLOCK_SIZE_M
        n = block_n * BLOCK_SIZE_N

        block_A = A[m : m + BLOCK_SIZE_M, :]
        block_B = B[:, n : n + BLOCK_SIZE_N]

        C[m : m + BLOCK_SIZE_M, n : n + BLOCK_SIZE_N] = block_A @ block_B

    return C

In [ ]:
def get_blocked_cuda_autotune_config():
    return [
        triton.Config({'BLOCK_SIZE_M': 128, 'BLOCK_SIZE_N': 256, 'BLOCK_SIZE_K': 64}, num_stages=3, num_warps=8),
        triton.Config({'BLOCK_SIZE_M': 64,  'BLOCK_SIZE_N': 256, 'BLOCK_SIZE_K': 32}, num_stages=4, num_warps=4),
        triton.Config({'BLOCK_SIZE_M': 128, 'BLOCK_SIZE_N': 128, 'BLOCK_SIZE_K': 32}, num_stages=4, num_warps=4),
        triton.Config({'BLOCK_SIZE_M': 128, 'BLOCK_SIZE_N': 64,  'BLOCK_SIZE_K': 32}, num_stages=4, num_warps=4),
        triton.Config({'BLOCK_SIZE_M': 64,  'BLOCK_SIZE_N': 128, 'BLOCK_SIZE_K': 32}, num_stages=4, num_warps=4),
        triton.Config({'BLOCK_SIZE_M': 128, 'BLOCK_SIZE_N': 32,  'BLOCK_SIZE_K': 32}, num_stages=4, num_warps=4),
        triton.Config({'BLOCK_SIZE_M': 64,  'BLOCK_SIZE_N': 32,  'BLOCK_SIZE_K': 32}, num_stages=5, num_warps=2),
        triton.Config({'BLOCK_SIZE_M': 32,  'BLOCK_SIZE_N': 64,  'BLOCK_SIZE_K': 32}, num_stages=5, num_warps=2),
    ]
@triton.autotune(configs = get_blocked_cuda_autotune_config(), key=['M', 'N', 'K'])

In [ ]:
@triton.jit
def blocked_matmul_kernel(
        A_ptr, B_ptr, C_ptr,
        M, N, K,
        stride_am, stride_ak,
        stride_bk, stride_bn,
        stride_cm, stride_cn,
        BLOCK_SIZE_M: tl.constexpr,
        BLOCK_SIZE_N: tl.constexpr,
        BLOCK_SIZE_K: tl.constexpr,
):
    pid = tl.program_id(axis=0)

    # get how many columns of blocks
    num_block_cols = tl.cdiv(N, BLOCK_SIZE_N)

    block_m = pid // num_block_cols     # which block of rows are we on?
    block_n = pid % num_block_cols      # which block of columns are we on?

    offs_m = block_m * BLOCK_SIZE_M + tl.arange(0, BLOCK_SIZE_M)
    offs_n = block_n * BLOCK_SIZE_N + tl.arange(0, BLOCK_SIZE_N)
    offs_k = tl.arange(0, BLOCK_SIZE_K)

    a_ptrs = A_ptr + (offs_m[:, None] * stride_am + offs_k[None, :] * stride_ak)
    b_ptrs = B_ptr + (offs_k[:, None] * stride_bk + offs_n[None, :] * stride_bn)

    accumulator = tl.zeros((BLOCK_SIZE_M, BLOCK_SIZE_N), dtype=tl.float32)

    for k in range(0, tl.cdiv(K, BLOCK_SIZE_K)):

        a_mask = (offs_m[:, None] < M) & (offs_k[None, :] < K - k * BLOCK_SIZE_K)
        a = tl.load(a_ptrs, mask=a_mask, other=0.0)

        b_mask = (offs_k[:, None] < K - k * BLOCK_SIZE_K) & (offs_n[None, :] < N)
        b = tl.load(b_ptrs, mask=b_mask, other=0.0)

        accumulator += tl.dot(a, b)      # block_size_m x block_size_n

        a_ptrs += BLOCK_SIZE_K * stride_ak
        b_ptrs += BLOCK_SIZE_K * stride_bk

    c = accumulator.to(A_ptr.dtype.element_ty)

    c_offsets = offs_m[:, None] * stride_cm + offs_n[None, :] * stride_cn

    c_mask = (offs_m[:, None] < M) & (offs_n[None, :] < N)

    tl.store(C_ptr + c_offsets, c, mask=c_mask)

In [ ]:
def blocked_matmul(a, b):

    # get dimensions
    M, K = a.shape
    K, N = b.shape

    # initialize output tensor with same dtype as input
    c = torch.empty((M, N), device=a.device, dtype=a.dtype)

    # define grid
    grid = lambda meta: (triton.cdiv(M, meta['BLOCK_SIZE_M']) * triton.cdiv(N, meta['BLOCK_SIZE_N']), )

    # launch kernel
    blocked_matmul_kernel[grid](
        a, b, c,
        M, N, K,
        a.stride(0), a.stride(1),
        b.stride(0), b.stride(1),
        c.stride(0), c.stride(1)
    )

    return c

In [ ]:
A = torch.randn(100, 500, device="cuda")
B = torch.randn(500, 1000, device="cuda")

print(blocked_matmul(A, B) - A@B)